# Module 5: Capstone, Multimodal Supplier Risk Intelligence

A factory in Vietnam catches fire on a Tuesday. By that afternoon it is in a Japanese trade publication and on a Chinese forum. By Thursday it reaches the English business press. By Friday your procurement team finds out.

This notebook builds the system that finds Tuesday.

## What you will do

1. Create one collection with three named vectors: dense text, sparse text, and CLIP image.
2. Ingest multilingual news and satellite tiles onto shared points.
3. Query in English and retrieve Japanese and Chinese sources, with no translation.
4. Query images with text, through CLIP's shared space.
5. Cluster signals into risk themes and use a cluster centroid as a query.
6. Run the analyst hybrid query, and break its filter on purpose to see why placement matters.

**Tip for Colab:** the first cells download `multilingual-e5-large` (about 2 GB) plus two small CLIP encoders, so give the setup a few minutes. Everything after that is fast.

Companion notebook to the [Module 5 lesson](https://qdrant.tech/course/beginners/module-5/).

## Setup

Three models, each chosen for a reason.

`intfloat/multilingual-e5-large` inherits 100 languages from XLM-RoBERTa and puts all of them in one vector space, which is the point of the whole system. `Qdrant/bm25` gives exact-token matching for supplier codes and ticker symbols. And a matched pair of CLIP encoders, one for images and one for text, puts pictures and words in a *second* shared space so a text query can retrieve a photo.

Nothing here needs a GPU or an API key.

In [ ]:
!pip install -q "qdrant-client[fastembed]" scikit-learn pillow 

In [ ]:
import warnings
import numpy as np
from qdrant_client import QdrantClient, models
from fastembed import TextEmbedding, SparseTextEmbedding, ImageEmbedding

warnings.filterwarnings("ignore")

TEXT_MODEL   = "intfloat/multilingual-e5-large"   # 1024-dim, 100 languages
SPARSE_MODEL = "Qdrant/bm25"                      # exact tokens
CLIP_VISION  = "Qdrant/clip-ViT-B-32-vision"      # 512-dim images
CLIP_TEXT    = "Qdrant/clip-ViT-B-32-text"        # 512-dim text, same space

text_model   = TextEmbedding(TEXT_MODEL)
sparse_model = SparseTextEmbedding(SPARSE_MODEL)
clip_vision  = ImageEmbedding(CLIP_VISION)
clip_text    = TextEmbedding(CLIP_TEXT)

TEXT_DIM = len(next(text_model.embed(["passage: warm up"])))
CLIP_DIM = len(next(clip_text.embed(["warm up"])))
print("text dim:", TEXT_DIM, "| clip dim:", CLIP_DIM)

text dim: 1024 | clip dim: 512


### The e5 prefixes are not optional

e5 was trained with `query:` on search text and `passage:` on stored text. FastEmbed does not add them for you, and if you skip them nothing errors.

Measuring the effect takes a little care, because the thing that matters is not how high a relevant document scores. It is how far a relevant document sits above an irrelevant one, since that gap is what ranking depends on. So we score one relevant passage and one irrelevant passage against the same query, both ways, and compare the margin.

In [ ]:
def unit(v):
    v = np.asarray(v, dtype=np.float32)
    return v / np.linalg.norm(v)

def embed_text(t, is_query=False):
    """e5 needs an explicit prefix. This is the only place we add it."""
    prefix = "query: " if is_query else "passage: "
    return unit(next(text_model.embed([prefix + t])))

def embed_raw(t):
    return unit(next(text_model.embed([t])))     # deliberately no prefix

question   = "factory fire halted production"
relevant   = "工場火災により生産が停止し、出荷の遅延が続いている。"
irrelevant = "The supplier reaffirmed full-year guidance and reported steady quarterly demand."

qv = embed_text(question, is_query=True)
with_rel, with_irr = float(qv @ embed_text(relevant)), float(qv @ embed_text(irrelevant))

qr = embed_raw(question)
raw_rel, raw_irr = float(qr @ embed_raw(relevant)), float(qr @ embed_raw(irrelevant))

print(f"with prefixes     relevant={with_rel:.3f}  irrelevant={with_irr:.3f}  margin={with_rel - with_irr:+.3f}")
print(f"without prefixes  relevant={raw_rel:.3f}  irrelevant={raw_irr:.3f}  margin={raw_rel - raw_irr:+.3f}")
print()
print("Skipping the prefixes RAISES both scores and NARROWS the gap between them.")
print("Absolute similarity is not the target. Separation is.")
print()
print("Note the scale too: e5 similarities compress into roughly 0.7 to 1.0,")
print("so never read one of these numbers as a percentage of relevance.")

with prefixes     relevant=0.804  irrelevant=0.719  margin=+0.085
without prefixes  relevant=0.823  irrelevant=0.753  margin=+0.070

Skipping the prefixes RAISES both scores and NARROWS the gap between them.
Absolute similarity is not the target. Separation is.

Note the scale too: e5 similarities compress into roughly 0.7 to 1.0,
so never read one of these numbers as a percentage of relevance.


## 1. One Collection, Not Three

Three modalities, and the instinct is three collections because that feels tidier.

A single event produces evidence in several modalities at once. That factory fire is a news article, a satellite image, and a line in an earnings call. Split by modality and you have split one event across three collections: every query hits all three and stitches results back together in your own code, and your filters get written three times.

Named vectors solve it. One point carries a dense text vector, a sparse one, and a CLIP image vector, and all of them share a single payload.

Two details below decide whether this works in production. The sparse config needs `modifier=models.Modifier.IDF`, or you are not scoring BM25. And every payload index is created **now**, before any data arrives, because Qdrant can only add filter-aware edges to the vector index for indexes that already exist when it is built. Here there are two dense vectors, so a late index means rebuilding two graphs.

`risk_score` is the one people forget, because nothing filters on it until Section 5. Miss it and the analyst query does not get slower, it fails: Qdrant Cloud enables strict mode by default, and strict mode rejects any query that filters an unindexed field.

In [ ]:
client = QdrantClient(":memory:")

client.create_collection(
    collection_name="supplier_signals",
    vectors_config={
        "text_dense": models.VectorParams(size=TEXT_DIM, distance=models.Distance.COSINE),
        "image":      models.VectorParams(size=CLIP_DIM, distance=models.Distance.COSINE),
    },
    sparse_vectors_config={
        "text_sparse": models.SparseVectorParams(modifier=models.Modifier.IDF),
    },
)

for field in ["supplier_id", "source_type", "language", "country", "facility_id"]:
    client.create_payload_index("supplier_signals", field_name=field,
                                field_schema=models.PayloadSchemaType.KEYWORD)

client.create_payload_index("supplier_signals", field_name="published_at",
                            field_schema=models.PayloadSchemaType.DATETIME)
client.create_payload_index("supplier_signals", field_name="risk_score",
                            field_schema=models.PayloadSchemaType.FLOAT)
client.create_payload_index("supplier_signals", field_name="cluster_id",
                            field_schema=models.PayloadSchemaType.INTEGER)

print("Collection created with 3 named vectors and 8 payload indexes, before ingestion.")

Collection created with 3 named vectors and 8 payload indexes, before ingestion.


## 2. Ingesting Multilingual Signals

The signals below stand in for a day's feed. Note the shape of the story: the Japanese and Chinese sources are reporting a shutdown, while the English sources are reporting routine quarterly news. That gap is what Section 6 goes looking for.

`source_type` is drawn from one fixed vocabulary, `news` or `satellite` here, because a filter written against a value nobody ingests returns nothing and warns you about nothing.

In [ ]:
signals = [
    # SUP-7291: local-language sources are ahead of the English ones
    dict(supplier_id="SUP-7291", language="ja", country="JP", source_type="news",
         published_at="2026-07-21T09:00:00Z", risk_score=0.88,
         text="工場火災により生産が停止し、出荷の遅延が続いている。復旧の見込みは立っていない。"),
    dict(supplier_id="SUP-7291", language="zh", country="CN", source_type="news",
         published_at="2026-07-21T14:00:00Z", risk_score=0.82,
         text="供应商工厂因火灾停产，交货期限推迟，客户已收到延误通知。"),
    dict(supplier_id="SUP-7291", language="ja", country="JP", source_type="news",
         published_at="2026-07-22T02:00:00Z", risk_score=0.71,
         text="労働争議が長引き、工場の稼働率が低下している。組合との交渉は難航している。"),
    dict(supplier_id="SUP-7291", language="vi", country="VN", source_type="news",
         published_at="2026-07-20T08:00:00Z", risk_score=0.64,
         text="Cang Hai Phong bi tac nghen, cac chuyen hang cua nha cung cap bi cham tre."),
    dict(supplier_id="SUP-7291", language="en", country="US", source_type="news",
         published_at="2026-07-22T07:15:00Z", risk_score=0.12,
         text="The supplier reaffirmed full-year guidance and reported steady quarterly demand."),
    dict(supplier_id="SUP-7291", language="en", country="GB", source_type="news",
         published_at="2026-07-22T11:00:00Z", risk_score=0.10,
         text="Analysts described the quarter as routine, with no change to the outlook for the group."),
    dict(supplier_id="SUP-7291", language="en", country="US", source_type="news",
         published_at="2026-07-19T16:00:00Z", risk_score=0.55,
         text="Ticker SUP7291.T slipped modestly on higher freight costs across the region."),
    # a second supplier, so tenant-style scoping has something to exclude
    dict(supplier_id="SUP-0002", language="zh", country="CN", source_type="news",
         published_at="2026-07-21T10:00:00Z", risk_score=0.79,
         text="另一家供应商的工厂发生停电，生产线暂时中断。"),
    dict(supplier_id="SUP-0002", language="en", country="DE", source_type="news",
         published_at="2026-07-18T09:00:00Z", risk_score=0.20,
         text="A European logistics operator announced a routine expansion of warehouse capacity."),
]

def to_sparse(text, is_query=False):
    emb = next(sparse_model.query_embed(text)) if is_query else next(sparse_model.embed([text]))
    return models.SparseVector(indices=emb.indices.tolist(), values=emb.values.tolist())

points = []
for i, s in enumerate(signals):
    points.append(models.PointStruct(
        id=i,
        vector={
            "text_dense":  embed_text(s["text"]).tolist(),   # passage: prefix
            "text_sparse": to_sparse(s["text"]),
        },
        payload={**s, "summary": s["text"][:60]},
    ))

client.upsert("supplier_signals", points=points)
print("ingested", client.count("supplier_signals").count, "text signals across",
      len({s["language"] for s in signals}), "languages")

ingested 9 text signals across 4 languages


## 3. Satellite Tiles Through CLIP

Real satellite imagery is not something a notebook can ship, so we generate four crude tiles instead. They are enough to show the mechanics and, as you will see, enough to show a real limitation too.

These points carry only the `image` vector. They live in the same collection as the text signals and share the same payload schema, which is the whole argument for named vectors.

In [ ]:
from PIL import Image, ImageDraw, ImageFilter
import os

os.makedirs("tiles", exist_ok=True)

def tile_smoke():
    im = Image.new("RGB", (224, 224), (70, 72, 78)); d = ImageDraw.Draw(im)
    d.rectangle([40, 150, 184, 224], fill=(48, 48, 52))
    for y, r in [(140, 20), (115, 28), (88, 36), (60, 44)]:
        d.ellipse([112 - r, y - r, 112 + r, y + r], fill=(190, 190, 195))
    return im.filter(ImageFilter.GaussianBlur(6))

def tile_farmland():
    im = Image.new("RGB", (224, 224), (94, 140, 60)); d = ImageDraw.Draw(im)
    for x in range(0, 224, 16):
        d.rectangle([x, 0, x + 8, 224], fill=(120, 168, 74))
    return im.filter(ImageFilter.GaussianBlur(1))

def tile_harbour():
    im = Image.new("RGB", (224, 224), (40, 90, 150)); d = ImageDraw.Draw(im)
    for x, y in [(30, 60), (120, 100), (70, 160), (150, 40)]:
        d.rectangle([x, y, x + 50, y + 18], fill=(210, 210, 215))
    return im.filter(ImageFilter.GaussianBlur(1))

def tile_fire():
    im = Image.new("RGB", (224, 224), (30, 20, 15)); d = ImageDraw.Draw(im)
    for r, c in [(90, (120, 30, 0)), (65, (200, 70, 0)), (40, (255, 150, 0)), (20, (255, 230, 120))]:
        d.ellipse([112 - r, 150 - r, 112 + r, 150 + r], fill=c)
    return im.filter(ImageFilter.GaussianBlur(8))

tiles = [
    ("smoke_plume", tile_smoke,    "SUP-7291", "FAC-01"),
    ("fire",        tile_fire,     "SUP-7291", "FAC-01"),
    ("harbour",     tile_harbour,  "SUP-7291", "FAC-02"),
    ("farmland",    tile_farmland, "SUP-0002", "FAC-09"),
]

paths = []
for name, fn, _, _ in tiles:
    p = f"tiles/{name}.png"
    fn().save(p)
    paths.append(p)

vecs = [unit(v).tolist() for v in clip_vision.embed(paths)]

client.upsert("supplier_signals", points=[
    models.PointStruct(
        id=100 + i,
        vector={"image": vec},
        payload=dict(supplier_id=sup, facility_id=fac, source_type="satellite",
                     language="n/a", country="VN",
                     published_at="2026-07-21T00:00:00Z",
                     risk_score=0.9 if name in ("fire", "smoke_plume") else 0.1,
                     summary=f"synthetic satellite tile: {name}"),
    )
    for i, (vec, (name, _, sup, fac)) in enumerate(zip(vecs, tiles))
])

print("collection now holds", client.count("supplier_signals").count, "points (text + imagery)")

collection now holds 13 points (text + imagery)


## 4. Two Kinds of Cross-Modal Query

### Text against images, via CLIP

The query text goes through CLIP's **text** encoder, not e5. That is the part people get wrong: each named vector is its own space, and a query only means something in the space it was embedded for. Sending an e5 vector at the `image` vector would return numbers, and they would be noise.

In [ ]:
def image_search(text, limit=4):
    qv = unit(next(clip_text.embed([text]))).tolist()   # CLIP text encoder
    return client.query_points("supplier_signals", query=qv, using="image",
                               limit=limit).points

for q in ["an orange fire burning", "ships in blue water at a port", "grey smoke rising into the sky"]:
    print(f"{q!r}")
    for r in image_search(q):
        print(f"   {r.score:.3f}  {r.payload['summary']}")
    print()

'an orange fire burning'
   0.267  synthetic satellite tile: fire
   0.209  synthetic satellite tile: farmland
   0.191  synthetic satellite tile: smoke_plume
   0.188  synthetic satellite tile: harbour

'ships in blue water at a port'
   0.209  synthetic satellite tile: harbour
   0.193  synthetic satellite tile: smoke_plume
   0.180  synthetic satellite tile: farmland
   0.141  synthetic satellite tile: fire

'grey smoke rising into the sky'
   0.211  synthetic satellite tile: fire
   0.204  synthetic satellite tile: farmland
   0.195  synthetic satellite tile: smoke_plume
   0.194  synthetic satellite tile: harbour



Two of those three are right, and the third is the useful one.

"an orange fire burning" and "ships in blue water at a port" retrieve the correct tiles. "grey smoke rising into the sky" does not: it prefers the fire tile.

That is not a bug in Qdrant or in the query. CLIP was trained on photographs, and these tiles are flat drawings made with a few ellipses. They sit outside the distribution the model learned, so its judgments get unreliable. Swap in real satellite imagery and this improves immediately.

The lesson generalizes past this notebook: cross-modal retrieval quality depends on your images resembling the model's training data, and the only way to know is to evaluate on your own.

### English query against local-language text

Now the capability the whole system exists for. The query is English, the corpus is not, and there is no translation step anywhere.

The filter has no prefetch above it, so `query_filter` is the correct placement here.

In [ ]:
def text_search(query_en, languages=None, supplier=None, limit=5):
    must = []
    if supplier:
        must.append(models.FieldCondition(key="supplier_id",
                                          match=models.MatchValue(value=supplier)))
    if languages:
        must.append(models.FieldCondition(key="language",
                                          match=models.MatchAny(any=languages)))
    return client.query_points(
        "supplier_signals",
        query=embed_text(query_en, is_query=True).tolist(),   # query: prefix
        using="text_dense",
        query_filter=models.Filter(must=must) if must else None,
        limit=limit,
    ).points

print("English query -> Japanese and Chinese sources only:\n")
for r in text_search("factory shutdown production halt",
                     languages=["ja", "zh"], supplier="SUP-7291"):
    p = r.payload
    print(f"   {r.score:.3f}  [{p['language']}] risk={p['risk_score']:.2f}  {p['summary']}")

English query -> Japanese and Chinese sources only:



   0.786  [zh] risk=0.82  供应商工厂因火灾停产，交货期限推迟，客户已收到延误通知。
   0.776  [ja] risk=0.88  工場火災により生産が停止し、出荷の遅延が続いている。復旧の見込みは立っていない。
   0.758  [ja] risk=0.71  労働争議が長引き、工場の稼働率が低下している。組合との交渉は難航している。


## 5. Clustering Signals Into Risk Themes

Clustering groups signals that describe the same underlying event even when they arrive in different languages from different sources.

Three practical points, all of them easy to get wrong.

**Page the scroll.** It returns a batch and an offset, and you keep going until the offset comes back empty. One capped call quietly clusters a busy supplier on partial data.

**Normalize before k-means.** These vectors are built for cosine similarity, but k-means measures Euclidean distance. Without normalizing you are partly clustering by vector length instead of direction.

**Drop the supplier filter to go wider.** A theme shared across suppliers shows up as one cluster pulling in signals from several of them at once.

In [ ]:
from sklearn.cluster import KMeans

def scroll_all(scroll_filter=None, page=64):
    """Page until the offset comes back empty."""
    out, offset = [], None
    while True:
        batch, offset = client.scroll("supplier_signals", scroll_filter=scroll_filter,
                                      with_vectors=True, limit=page, offset=offset)
        out.extend(batch)
        if offset is None:
            return out

def dense_matrix(points):
    """Unit-normalized text_dense vectors, with the ids they belong to."""
    ids, vecs = [], []
    for p in points:
        if p.vector and "text_dense" in p.vector:      # image-only points have none
            ids.append(p.id)
            vecs.append(p.vector["text_dense"])
    if not vecs:
        return [], None
    arr = np.asarray(vecs, dtype=np.float32)
    arr /= np.linalg.norm(arr, axis=1, keepdims=True)
    return ids, arr

text_only = models.Filter(must=[models.FieldCondition(
    key="source_type", match=models.MatchValue(value="news"))])

pts = scroll_all(text_only)
ids, arr = dense_matrix(pts)
print(f"scrolled {len(pts)} points, {len(ids)} carry a text vector")

labels = KMeans(n_clusters=3, n_init=10, random_state=42).fit_predict(arr)

# One set_payload call per cluster, not one per point
for label in sorted({int(l) for l in labels}):
    client.set_payload("supplier_signals", payload={"cluster_id": label},
                       points=[i for i, l in zip(ids, labels) if int(l) == label])

by_id = {p.id: p.payload for p in pts}
for label in sorted({int(l) for l in labels}):
    print(f"\ncluster {label}:")
    for i, l in zip(ids, labels):
        if int(l) == label:
            p = by_id[i]
            print(f"   [{p['language']}] risk={p['risk_score']:.2f}  {p['summary'][:52]}")

scrolled 9 points, 9 carry a text vector

cluster 0:
   [vi] risk=0.64  Cang Hai Phong bi tac nghen, cac chuyen hang cua nha

cluster 1:
   [en] risk=0.12  The supplier reaffirmed full-year guidance and repor
   [en] risk=0.10  Analysts described the quarter as routine, with no c
   [en] risk=0.55  Ticker SUP7291.T slipped modestly on higher freight 
   [en] risk=0.20  A European logistics operator announced a routine ex

cluster 2:
   [ja] risk=0.88  工場火災により生産が停止し、出荷の遅延が続いている。復旧の見込みは立っていない。
   [zh] risk=0.82  供应商工厂因火灾停产，交货期限推迟，客户已收到延误通知。
   [ja] risk=0.71  労働争議が長引き、工場の稼働率が低下している。組合との交渉は難航している。
   [zh] risk=0.79  另一家供应商的工厂发生停电，生产线暂时中断。


Notice that the clusters cross language boundaries. The Japanese fire report and the Chinese shutdown report land together because they describe the same event, which is exactly what a single multilingual vector space buys you.

### The centroid as a query

A cluster centroid is just another vector, so you can hand it straight back to Qdrant as a query and pull in more of the same theme. That is how you go from "here are today's clusters" to "find everything that looks like this emerging story".

In [ ]:
target = int(labels[0])
member_rows = [i for i, l in zip(ids, labels) if int(l) == target]
centroid = arr[[ids.index(i) for i in member_rows]].mean(axis=0)
centroid = centroid / np.linalg.norm(centroid)

print(f"querying with the centroid of cluster {target}:\n")
for r in client.query_points("supplier_signals", query=centroid.tolist(),
                             using="text_dense", limit=5).points:
    p = r.payload
    mark = "  <- in the cluster" if r.id in member_rows else ""
    print(f"   {r.score:.3f}  [{p['language']}] {p['summary'][:48]}{mark}")

querying with the centroid of cluster 2:

   0.962  [zh] 供应商工厂因火灾停产，交货期限推迟，客户已收到延误通知。  <- in the cluster
   0.961  [zh] 另一家供应商的工厂发生停电，生产线暂时中断。  <- in the cluster
   0.957  [ja] 工場火災により生産が停止し、出荷の遅延が続いている。復旧の見込みは立っていない。  <- in the cluster
   0.947  [ja] 労働争議が長引き、工場の稼働率が低下している。組合との交渉は難航している。  <- in the cluster
   0.798  [vi] Cang Hai Phong bi tac nghen, cac chuyen hang cua


## 6. The Analyst Query, and the Mistake That Fails Silently

The query analysts actually run: hybrid retrieval over dense and sparse, scoped to one supplier, elevated risk only.

One detail decides whether it works. The filter goes **inside each prefetch**, not on the outer query. Prefetches run first and the outer query is applied to their results, so a top-level filter arrives too late: both retrievers search every supplier and every risk level, and the filter only trims the fused list at the end.

Both versions run below. Watch the `risk` and `supplier` columns in the broken one.

In [ ]:
def analyst_search(query_text, supplier, min_risk=0.5, limit=5, broken=False):
    dense_q  = embed_text(query_text, is_query=True).tolist()
    sparse_q = to_sparse(query_text, is_query=True)
    risk_filter = models.Filter(must=[
        models.FieldCondition(key="supplier_id", match=models.MatchValue(value=supplier)),
        models.FieldCondition(key="risk_score", range=models.Range(gte=min_risk)),
    ])
    if broken:
        return client.query_points(
            "supplier_signals",
            prefetch=[
                models.Prefetch(query=dense_q,  using="text_dense",  limit=50),
                models.Prefetch(query=sparse_q, using="text_sparse", limit=50),
            ],
            query=models.RrfQuery(rrf=models.Rrf()),
            query_filter=risk_filter,          # too late
            limit=limit,
        ).points
    return client.query_points(
        "supplier_signals",
        prefetch=[
            models.Prefetch(query=dense_q,  using="text_dense",
                            filter=risk_filter, limit=50),
            models.Prefetch(query=sparse_q, using="text_sparse",
                            filter=risk_filter, limit=50),
        ],
        query=models.RrfQuery(rrf=models.Rrf()),
        limit=limit,
    ).points

def report(title, rows):
    print(title)
    if not rows:
        print("   (no results)")
    for r in rows:
        p = r.payload
        print(f"   {r.score:.4f}  {p['supplier_id']}  risk={p['risk_score']:.2f}  "
              f"[{p['language']}] {p['summary'][:42]}")
    print()

report("CORRECT  filter inside each prefetch:",
       analyst_search("production halt at the factory", "SUP-7291"))
report("BROKEN   same filter at the top level:",
       analyst_search("production halt at the factory", "SUP-7291", broken=True))

CORRECT  filter inside each prefetch:
   0.5000  SUP-7291  risk=0.82  [zh] 供应商工厂因火灾停产，交货期限推迟，客户已收到延误通知。
   0.3333  SUP-7291  risk=0.88  [ja] 工場火災により生産が停止し、出荷の遅延が続いている。復旧の見込みは立っていない。
   0.2500  SUP-7291  risk=0.55  [en] Ticker SUP7291.T slipped modestly on highe
   0.2000  SUP-7291  risk=0.71  [ja] 労働争議が長引き、工場の稼働率が低下している。組合との交渉は難航している。
   0.1667  SUP-7291  risk=0.64  [vi] Cang Hai Phong bi tac nghen, cac chuyen ha



BROKEN   same filter at the top level:
   0.5000  SUP-0002  risk=0.79  [zh] 另一家供应商的工厂发生停电，生产线暂时中断。
   0.3333  SUP-7291  risk=0.82  [zh] 供应商工厂因火灾停产，交货期限推迟，客户已收到延误通知。
   0.2500  SUP-7291  risk=0.88  [ja] 工場火災により生産が停止し、出荷の遅延が続いている。復旧の見込みは立っていない。
   0.2000  SUP-7291  risk=0.55  [en] Ticker SUP7291.T slipped modestly on highe
   0.1667  SUP-7291  risk=0.71  [ja] 労働争議が長引き、工場の稼働率が低下している。組合との交渉は難航している。



The broken version returns rows that violate the filter, with no error and no warning. On a real corpus, where one supplier is a thousandth of the collection instead of most of it, the same mistake usually returns nothing at all, and an analyst concludes there is no news.

The rule, one more time. No prefetch, use `query_filter`. Prefetch, put the filter in every prefetch.

## 7. Reading the Gap

The payoff. Run the same query twice, once scoped to English sources and once to Japanese and Chinese, and compare.

This is not a different algorithm. It is the same query and the same vector space with a different `language` filter.

In [ ]:
query = "factory shutdown, production halted, delivery delays"

report("ENGLISH sources only:",
       text_search(query, languages=["en"], supplier="SUP-7291"))
report("JAPANESE and CHINESE sources only:",
       text_search(query, languages=["ja", "zh"], supplier="SUP-7291"))

ENGLISH sources only:
   0.7789  SUP-7291  risk=0.55  [en] Ticker SUP7291.T slipped modestly on highe
   0.7239  SUP-7291  risk=0.12  [en] The supplier reaffirmed full-year guidance
   0.6827  SUP-7291  risk=0.10  [en] Analysts described the quarter as routine,

JAPANESE and CHINESE sources only:
   0.8059  SUP-7291  risk=0.82  [zh] 供应商工厂因火灾停产，交货期限推迟，客户已收到延误通知。
   0.7946  SUP-7291  risk=0.88  [ja] 工場火災により生産が停止し、出荷の遅延が続いている。復旧の見込みは立っていない。
   0.7731  SUP-7291  risk=0.71  [ja] 労働争議が長引き、工場の稼働率が低下している。組合との交渉は難航している。



The English sources are reporting reaffirmed guidance and a routine quarter. The Japanese and Chinese sources are reporting a fire, halted production, and delayed deliveries, and they were published earlier.

That gap is the early warning, and finding it needed no translation pipeline, no separate per-language index, and no second database. One collection, one multilingual model, one filter.

## Course Close

Look at what is in that collection. Text, sparse tokens, and imagery on shared points. Filters that hold. Hybrid retrieval. Clustering. Cross-language and cross-modal search.

Six primitives got you here: collection, point, vector, payload, index, query.

Module 1 asked why keyword search misses. Module 2 opened up the vector. Module 3 put dense and sparse together. Module 4 turned that into a design you could defend. This module ran the whole thing on three modalities at once.

The next system someone hands you is these six, arranged differently.

### Your turn

Swap the synthetic tiles for real satellite imagery and rerun Section 4, then check whether the smoke query starts behaving.

Add a `tenant_id` field with `is_tenant=True` and scope every query to one desk, the way Module 4 did.

Then try `models.Rrf(weights=[3.0, 1.0])` in the analyst query to favour dense over sparse, and see which retriever your data actually prefers.